# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, sum as _sum, expr, sha2, concat_ws, regexp_replace, when

# Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.lc_loans_raw")

# Data Transformations

## tạo value cho cột id và memberid

In [0]:
df = df.withColumn("id", expr("uuid()"))

borrower_traits = [
    "emp_title",      
    "annual_inc",     
    "home_ownership", 
    "addr_state"     
]

df = df.withColumn(
    "member_id", 
    sha2(concat_ws("_", *[col(c) for c in borrower_traits]), 256)
)

## Loại bỏ các cột có giá trị null chiếm > 80% và các cột không cần thiết cho ml & report

In [0]:
total_rows = df.count()
threshold = 0.8

null_counts_row = df.select([
    _sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
]).first().asDict()

cols_to_drop = [
    c for c, null_count in null_counts_row.items() 
    if (null_count / total_rows) > threshold
]

print(f"Tổng số dòng: {total_rows}")
print(f"Số lượng cột bị xóa (> {threshold*100}% Null): {len(cols_to_drop)}")
print(f"Danh sách các cột bị xóa:\n{cols_to_drop}")

df = df.drop(*cols_to_drop)
cols_to_drop_round_2 = [
    "title", "zip_code", "initial_list_status", "funded_amnt_inv",
    "policy_code", "pymnt_plan", "hardship_flag", "debt_settlement_flag", 
    "application_type"
]

df = df.drop(*cols_to_drop_round_2)

## Deduplication

In [0]:
df = df.dropDuplicates(["id"])

## làm sạch data cột term,empt_lenght

In [0]:
df = df.withColumn(
    "term", 
    regexp_replace(col("term"), " months", "")
).withColumn(
    "emp_length_clean", 
    when(col("emp_length").contains("< 1"), "0")
    .otherwise(regexp_replace(col("emp_length"), "[^0-9]", ""))
).withColumn(
    "emp_length", 
    expr("try_cast(emp_length_clean AS INT)")
).drop("emp_length_clean") 

## xử lý dữ liệu null

In [0]:
#cột emp_length
median_row = df.selectExpr("percentile_approx(emp_length, 0.5)").collect()[0]
median_value = int(median_row[0]) 
#cột mths_since_last_delinq
df = df.withColumn(
    "has_prior_delinq",
    when(col("mths_since_last_delinq").isNull(), 0).otherwise(1)
)
#cột next_pymnt_d 
df = df.withColumn(
    "next_pymnt_d",
    when(
        (col("next_pymnt_d").isNull()) & (col("loan_status").isin("Fully Paid", "Charged Off")), 
        "Closed"
    )
    .when(
        col("next_pymnt_d").isNull(), 
        "Unknown"
    )
    .otherwise(col("next_pymnt_d"))
)
#cột mths_since_last_major_derog
df = df.withColumn(
    "has_major_derog",
    when(col("mths_since_last_major_derog").isNull(), 0)
    .otherwise(1)
).withColumn(
    "mths_since_last_major_derog",
    when(col("mths_since_last_major_derog").isNull(), 999)
    .otherwise(col("mths_since_last_major_derog"))
)
time_columns = [
    "mths_since_recent_bc",
    "mths_since_recent_bc_dlq",
    "mths_since_recent_inq",
    "mths_since_recent_revol_delinq"
]
fill_values = {c: 999 for c in time_columns}
df = df.fillna(fill_values)
df = df.fillna({"emp_length": median_value,
                "emp_title": "other",
                "mths_since_last_delinq": -1,
                "il_util": -1.0,
                "bc_open_to_buy": 0,
                "num_tl_120dpd_2m": 0,
                "bc_util": -1.0,
                "percent_bc_gt_75": -1.0,
                "mths_since_rcnt_il": 999,
                "mo_sin_old_il_acct": -1
})

In [0]:
display(df)